# Lab 7: Let's Replicate (part of) a Machine Learning Paper! 📝

Can measurements from ordinary environmental sensors tell us whether an office
is occupied? And does giving a model more sensors always make it better?

In this lab, you will reproduce part of a published machine-learning study:

> Luis M. Candanedo and Véronique Feldheim, “Accurate occupancy detection of
> an office room from light, temperature, humidity and CO₂ measurements using
> statistical learning models,” *Energy and Buildings* 112 (2016), 28–39.

The authors measured temperature, relative humidity, light, CO₂, and humidity
ratio about once per minute. They determined the true occupancy from
timestamped photographs. Their data are available through the
[UCI Machine Learning Repository](https://doi.org/10.24432/C5X01N), and their
[analysis code is public](https://github.com/LuisM78/Occupancy-detection-data).

You will reuse the authors' dataset.
You will use scikit-learn pipelines, `GridSearchCV`, and three classifiers you
have already seen: logistic regression, k-nearest neighbors, and random forest.
(In the paper, the authors explore many models, including random forests. But
from I can tell, they didn't try k-NN or logistic regression.)

You should complete this entire lab so that all tests pass.


In [ ]:
in_colab = "google.colab" in str(get_ipython())
if in_colab:
    !pip install otter-grader==6.1.6

from pathlib import Path
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

path = 'labs/lab07/build/student'
if in_colab and not Path('data').exists():
    !wget -q -O /content/course.zip https://github.com/dsc-courses/cosmos-ml-cluster-2026/archive/refs/heads/main.zip
    with zipfile.ZipFile('/content/course.zip') as course_zip:
        archive_prefix = f'cosmos-ml-cluster-2026-main/{path}/'
        members = [name for name in course_zip.namelist() if name.startswith(archive_prefix)]
        course_zip.extractall('/content/course-assets', members)
    source_path = Path('/content/course-assets') / archive_prefix
    shutil.copytree(source_path / 'data', 'data', dirs_exist_ok=True)
    shutil.copytree(source_path / 'tests', 'tests', dirs_exist_ok=True)

import otter
grader = otter.Notebook()

plt.style.use('seaborn-v0_8-colorblind')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_colwidth', 120)


## 1. Understand the experiment 🔎

### Why detect occupancy?

Buildings use a great deal of energy heating, cooling, ventilating, and
lighting rooms. If a control system knows that a room is empty, it can turn
down those systems instead of conditioning an unused space. Occupancy
information can also help keep an occupied room comfortable, show whether a
shared room is available, or alert staff when a supposedly empty area is in
use.

One natural question to ask is: **why not simply point a camera at the room?**
A camera can be an effective way to
detect people, but continuously sending or storing office images can create
serious privacy concerns. Images also require more storage, bandwidth, and
computation than a handful of sensor readings, and camera predictions may fail
when a person is blocked from view or the lighting is poor. The goal of this
study is therefore to learn whether small, non-visual environmental
measurements can provide a useful occupancy signal. The researchers did use a
camera to create the ground-truth labels for this experiment, but the fitted
models use no images and would not require a camera when deployed.

### How were the data collected?

Sensors in one office recorded temperature, relative humidity, light level,
and CO₂ concentration approximately once per minute. `HumidityRatio` was then
derived from temperature and relative humidity. For the ground truth,
a timestamped photograph was
taken every minute and a human used the photos to label whether the room was occupied at
that time.

In the dataset, `Occupancy = 1` means at least one
person was present, and `Occupancy = 0` means nobody was present. (It does not
tell us how many people were in the office, so even if there were somehow 1,000 people in the office,
the label would still be `1`.)

People can affect several sensors. We exhale CO₂ and water vapor and give off
heat, so CO₂, humidity, and temperature may rise while a room is occupied.
People may also switch on the lights. These are useful clues, but aren't
perfect: for example, ventilation, an open door, outdoor weather, and
automatic lighting can also change the sensor readings.

### The Data

The authors provide three consecutive *measurement periods*, with small gaps
between them:

| File | Dates | Rows | Role in the paper |
| --- | --- | ---: | --- |
| `occupancy_test_closed.csv` | Feb. 2–4, 2015 | 2,665 | Earlier external test period; door mostly closed during occupancy |
| `occupancy_train.csv` | Feb. 4–10, 2015 | 8,143 | Training and cross-validation period |
| `occupancy_test_open.csv` | Feb. 11–18, 2015 | 9,752 | Later external test period; door mostly open during occupancy |

Thus, the training data we have is the middle file in chronological order. Keeping the same
partition makes our results comparable with the authors' results. The
two external files then test whether a model transfers backward to an earlier
similar period and forward to a later period with different conditions; in this
case, the office door was left open.

The target `Occupancy` is 1 when the room was occupied and 0 otherwise. The
sensor columns are:

| Column | Meaning | Unit |
| --- | --- | --- |
| `Temperature` | Air temperature | °C |
| `Humidity` | Relative humidity | % |
| `Light` | Illuminance | Lux |
| `CO2` | Carbon dioxide concentration | ppm |
| `HumidityRatio` | Water vapor divided by dry air | kg/kg |

In [ ]:
# Run this cell to load the data
training = pd.read_csv('data/occupancy_train.csv')

sensor_columns = [
    'Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio'
]
X_train = training[['date', *sensor_columns]]
y_train = training['Occupancy']

training.head()


**Question 1.1.** Create the following five variables to audit the
training period:

- `training_shape`: a `tuple` of two `int`s containing the number of
  rows and columns in `training`.
- `training_start`: a pandas `Timestamp` containing the earliest
  timestamp in `training`.
- `training_end`: a pandas `Timestamp` containing the latest timestamp
  in `training`.
- `num_missing`: an `int` containing the total number of missing values
  in `training`.
- `num_duplicate_ids`: an `int` containing the number of duplicated
  values in the `id` column.


In [ ]:
...

print(f'Rows and columns: {training_shape}')
print(f'Training period: {training_start} through {training_end}')
print(f'Missing values: {num_missing}')
print(f'Duplicate IDs: {num_duplicate_ids}')


In [ ]:
grader.check("q1_1")

**Question 1.2.** Create `occupancy_rates`, a pandas `Series` containing
the proportion of training rows in each class. Its index should be the
integers `0` and `1`, in that order, and its values should be `float`s.

Now imagine a simple classifier: rather than looking at any of the
features, this classifier always predicts the **most common class**
(`0` or `1`) in the training data. Create `majority_baseline`, a `float`
containing the accuracy of this classifier, computed from
`occupancy_rates`.

We often start by computing this baseline because a useful fitted model
should do at least as well. Otherwise, we would be better off always
making the most common prediction.


In [ ]:
...

display(occupancy_rates.rename('proportion'))
print(f'Majority-class baseline accuracy: {majority_baseline:.1%}')


In [ ]:
grader.check("q1_2")

### Sensor measurements over time

Run the cell below to reproduce the paper's basic time-series exploration. The
gray shading marks times when the office was occupied.

In [ ]:
plot_data = training.copy()
plot_data['date'] = pd.to_datetime(plot_data['date'])

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
plot_columns = [*sensor_columns, 'Occupancy']

for axis, column in zip(axes.flat, plot_columns):
    axis.plot(plot_data['date'], plot_data[column], linewidth=1)
    axis.set_ylabel(column)
    for start in plot_data.loc[plot_data['Occupancy'].diff().eq(1), 'date']:
        later = plot_data.loc[
            (plot_data['date'] >= start) & plot_data['Occupancy'].eq(0), 'date'
        ]
        end = later.iloc[0] if len(later) else plot_data['date'].iloc[-1]
        axis.axvspan(start, end, color='gray', alpha=0.12)

axes[-1, 0].tick_params(axis='x', rotation=25)
axes[-1, 1].tick_params(axis='x', rotation=25)
fig.suptitle('Training-period sensor readings (gray = occupied)', y=1.02)
fig.tight_layout()


**Pause and interpret.** Which sensor most visibly separates occupied from
unoccupied periods? Which measurements seem to respond slowly after somebody
enters or leaves? Why might `HumidityRatio` contain less new information than
its name suggests?


## 2. Feature engineering 🕒

The authors did not pass the raw timestamp into their models
(can you figure out why this wouldn't be useful?).
Instead, they extracted the number of seconds since midnight (`NSM`) and whether the date was a weekday
or weekend (`WeekStatus`). We will recreate those two features.

**Question 2.1.** Complete the function `make_occupancy_features`. It
should take three parameters:

- `data`: a pandas `DataFrame` containing a `date` column and sensor
  columns.
- `sensor_columns`: a `list` of `str`s naming the sensor columns to
  extract from `data`.
- `include_time`: a `bool` indicating whether to derive time features.

The function should return a **new pandas `DataFrame`** with the same
rows, index, and row order as `data`, but containing only the columns
named in `sensor_columns`. The columns should appear in the same order
as the strings in `sensor_columns`.

If `include_time` is `True`, the returned DataFrame should also contain
these two columns:

- `seconds_since_midnight`: an integer column containing the number of
  seconds elapsed since midnight for each row's timestamp. For example,
  midnight is 0 and noon is 43,200.
- `is_weekend`: a Boolean column that is `True` when the timestamp falls
  on Saturday or Sunday and `False` otherwise.

The returned DataFrame should never contain `id`, the raw `date`, or
`Occupancy`, and the function should not modify `data`.


In [ ]:
...

make_occupancy_features(X_train.head(), sensor_columns, include_time=True)


In [ ]:
grader.check("q2_1")

**Question 2.2.** Create these two objects:

- `time_transformer`: a scikit-learn `FunctionTransformer` that calls
  `make_occupancy_features` with all five sensor columns and
  `include_time=True`.
- `transformed_example`: a pandas `DataFrame` produced by applying
  `time_transformer` to the first three rows of `X_train`. It should
  have 3 rows and 7 columns: the five sensors followed by
  `seconds_since_midnight` and `is_weekend`.


In [ ]:
...

transformed_example

In [ ]:
grader.check("q2_2")

### Feature recipes

Although the authors collected data from multiple sensors, it's also useful to ask:
**which measurements are actually useful for prediction?**
Sensors cost money to purchase, install, calibrate, and power. Some
features may also be redundant or noisy. For example, humidity ratio is derived
from two other measurements, so giving a model every available column might not
improve its predictions.

What about using time? Seconds since midnight and
weekday/weekend status can capture a regular office schedule, which may improve
accuracy even when sensor readings are ambiguous. But a model that relies on
the schedule may fail during holidays, unusual work hours, or in a different
office. Comparing recipes with and without time lets us measure exactly how
useful time is to the model.

Since the authors tried many combinations of features, we've stored the
author's feature combinations in `feature_recipes.csv`.

**Question 2.3.** Create these two objects:

- `feature_recipes_df`: a pandas `DataFrame` loaded from
  `data/feature_recipes.csv`. It should have the columns `recipe`,
  `sensor_columns`, and `include_time`. Convert every value in
  `sensor_columns` from JSON text into a Python `list` of `str`s using
  `json.loads`; `include_time` should contain Boolean values.
- `feature_recipes`: a `dict` keyed by recipe-name strings. Each value
  should be another `dict` with a `sensor_columns` list and an
  `include_time` Boolean, ready to pass to
  `make_occupancy_features`.

*Hint:* if `feature_recipes_df` has the required types,
`feature_recipes_df.set_index('recipe').to_dict(orient='index')`
performs the dictionary conversion for you.


In [ ]:
...

feature_recipes_df

In [ ]:
grader.check("q2_3")

## 3. Tune three classifiers ⚙️

The authors used accuracy to compare models.
But notice that the `occupied` class (the `1` label) is less common than the `0` label.
To account for this *class imbalance*, we will use
`StratifiedKFold` so every fold has a similar ratio of `0` and `1`.

Every pipeline will use this order:

1. Select and engineer features with `FunctionTransformer`.
2. Standardize them with `StandardScaler`.
3. Fit the classifier.

Recall that standardization is super important for logistic regression and kNN!
A random forest won't do better or worse with standardization, but using the
same preprocessing structure makes the code a bit easier to write so we'll
standardize anyway.


**Question 3.1.** Create `folds`, a `StratifiedKFold` object with five
folds, `shuffle=True`, and `random_state=0`. Then complete
`model_specs`, a `dict` describing all three classifiers.

The keys of `model_specs` are model-name strings. Each value is another
`dict` with exactly two entries:

- `'estimator'` must contain an **unfitted scikit-learn classifier
  object**, such as `LogisticRegression(...)`.
- `'parameter_grid'` must contain a **`dict` mapping `str` parameter
  names to `list`s of candidate values**. Because the classifier lives
  inside a pipeline, each parameter name begins with the automatically
  generated step name, followed by two underscores—for example,
  `'logisticregression__C'`.

The logistic-regression entry is completed as an example. Add:

- `knn`: an unfitted `KNeighborsClassifier` with `n_jobs=-1`; search 3,
  5, 11, 25, or 51 neighbors and `'uniform'` or `'distance'` weights.
- `random_forest`: an unfitted `RandomForestClassifier` with 200 trees,
  `random_state=0`, and `n_jobs=1`; search maximum depth `None` or 8,
  minimum leaf size 1 or 5, and maximum features `'sqrt'` or `None`.


In [ ]:
...


In [ ]:
model_specs = {
    # We gave you the code for logistic regression as a starting point
    'logistic_regression': {
        'estimator': LogisticRegression(max_iter=2000, random_state=0),
        'parameter_grid': {
            'logisticregression__C': [0.01, 0.1, 1, 10],
        },
    },
    ...
}


In [ ]:
grader.check("q3_1")

**Question 3.2.** Complete the function `make_search`. It should take
three parameters:

- `estimator`: an unfitted scikit-learn classifier object.
- `parameter_grid`: a `dict` mapping pipeline parameter-name strings to
  `list`s of candidate values.
- `recipe`: a `dict` with a `sensor_columns` list and an `include_time`
  Boolean.

The function should return an **unfitted `GridSearchCV` object**. Its
estimator should be a pipeline containing, in order, a
`FunctionTransformer`, a `StandardScaler`, and a clone of `estimator`.
Configure the search with `folds`, accuracy scoring, `n_jobs=-1`, and
`return_train_score=True`.


In [ ]:
...


In [ ]:
grader.check("q3_2")

### Establish a common baseline

First, let's fit all three classifiers using all available features: the
five sensors and the two timestamp features. 


**Question 3.3.** For every item in `model_specs`, create a grid search
using the `all_plus_time` recipe and fit it on `X_train` and `y_train`.
Create these two objects:

- `baseline_searches`: a `dict` mapping each model-name `str` to its
  fitted `GridSearchCV` object.
- `baseline_results`: a pandas `DataFrame` with one row per model and
  the columns `model`, `cv_accuracy`, `cv_std`, `train_accuracy`, and
  `best_parameters`. The first four result values should be strings or
  floats as appropriate, and each `best_parameters` value should be a
  `dict`.


In [ ]:
...

baseline_results.sort_values('cv_accuracy', ascending=False)

In [ ]:
grader.check("q3_3")

**Pause and interpret.** Which model has the highest cross-validation accuracy?
Are the gaps large enough to matter? Which model has the largest gap between
training and CV accuracy? 


## 4. Reproduce the feature-selection study 🧪

Now, let's reproduce the author's feature combinations. Remember that the authors
wanted to see whether they could make the model simpler by removing features while
still keeping a high accuracy.

**Question 4.1.** Fit a grid search for every feature recipe and model
using `feature_recipes`. Create these two objects:

- `searches`: a `dict` whose keys are `(recipe_name, model_name)` tuples
  of strings and whose values are fitted `GridSearchCV` objects. Reuse
  the fitted object from `baseline_searches` for each `all_plus_time`
  entry.
- `cv_results`: a pandas `DataFrame` with one row per recipe/model pair
  and the columns `recipe`, `model`, `num_features`, `cv_accuracy`,
  `cv_std`, `train_accuracy`, and `best_parameters`. `num_features`
  should be an integer count of the features passed to the classifier,
  and `best_parameters` should contain dictionaries.


In [ ]:
...

cv_results.pivot(index='recipe', columns='model', values='cv_accuracy').style.format('{:.2%}')


In [ ]:
grader.check("q4_1")

### Freeze the choices

For each classifier, we will select the feature recipe with the highest CV
accuracy. Once these choices are recorded, we will not refit or retune anything
after seeing external test results.


**Question 4.2.** Create these two objects using only cross-validation
accuracy:

- `frozen_choices`: a pandas `DataFrame` containing exactly one row for
  each model—the row of `cv_results` with that model's highest
  `cv_accuracy`. Sort it by model name and reset its index.
- `frozen_searches`: a `dict` mapping each model-name `str` to the
  already-fitted `GridSearchCV` selected by `frozen_choices`.


In [ ]:
...

frozen_choices[['model', 'recipe', 'num_features', 'cv_accuracy', 'best_parameters']]

In [ ]:
grader.check("q4_2")

Before continuing, write down a prediction: which frozen model will transfer
best to the two external periods? Will the winner be the same for both?


**Question 4.3.** A fitted random forest records a **feature
importance** for every input column. In scikit-learn, this measures how
much the trees' splits using that feature reduced impurity, averaged
across the forest. The values are nonnegative and sum to 1, so a larger
value means the forest relied more heavily on that feature.

The classifier is nested inside the selected pipeline. Retrieve the
fitted pipeline with:

```python
frozen_forest = frozen_searches['random_forest'].best_estimator_
```

Then retrieve the fitted classifier and its one-dimensional NumPy
importance array with:

```python
forest_classifier = frozen_forest.named_steps['randomforestclassifier']
forest_classifier.feature_importances_
```

Create `forest_importances`, a pandas `Series` of `float`s. Its index
should contain the transformed feature-name strings, and its values
should contain the corresponding importances in descending order. Then
use this Series to make a horizontal bar chart.


In [ ]:
...

forest_importances

In [ ]:
grader.check("q4_3")

**Pause and interpret.** Is light important to the selected forest? Does feature
importance prove that a sensor causes occupancy? Why should the importance of
`HumidityRatio` be interpreted cautiously?


## 5. Evaluate on two external periods 🚪

Every modeling choice is now frozen. We can finally load the two test periods.
The first was measured under conditions similar to training, with the office
door mostly closed during occupancy. In the second, the door was mostly open.
An open door can change how temperature, humidity, and CO₂ behave.

Remember that the first test period occurs *before* the training period,
while the second occurs after it.

In [ ]:
test_closed = pd.read_csv('data/occupancy_test_closed.csv')
test_open = pd.read_csv('data/occupancy_test_open.csv')

X_test_closed = test_closed[['date', *sensor_columns]]
y_test_closed = test_closed['Occupancy']
X_test_open = test_open[['date', *sensor_columns]]
y_test_open = test_open['Occupancy']

test_summary = pd.DataFrame({
    'rows': [len(test_closed), len(test_open)],
    'start': [pd.to_datetime(test_closed['date']).min(), pd.to_datetime(test_open['date']).min()],
    'end': [pd.to_datetime(test_closed['date']).max(), pd.to_datetime(test_open['date']).max()],
    'occupied_rate': [y_test_closed.mean(), y_test_open.mean()],
}, index=['closed-door period', 'open-door period'])
test_summary


**Question 5.1.** Evaluate each of the three chosen models on both
external periods. Create `frozen_test_results`, a pandas `DataFrame`
with exactly one row per model and the columns `model`, `recipe`,
`cv_accuracy`, `closed_accuracy`, and `open_accuracy`. The model and
recipe values should be strings; the three accuracy values should be
floats.

Do not call `.fit` anywhere in this question.


In [ ]:
...

frozen_test_results.style.format({column: '{:.2%}' for column in frozen_test_results.columns if 'accuracy' in column})

In [ ]:
grader.check("q5_1")

### A paper-style comparison table

Let's now recreate the final table of results from the paper using our models.


**Question 5.2.** Evaluate every fitted search in `searches` on both
test periods. Create `external_results`, a pandas `DataFrame` with one
row for each of the 36 recipe/model pairs and the columns `recipe`,
`model`, `cv_accuracy`, `closed_accuracy`, `open_accuracy`, and
`mean_external_accuracy`. Recipe and model values should be strings;
all accuracy values should be floats.

Sort the DataFrame from largest to smallest `mean_external_accuracy`
and reset its index. Do not call `.fit` anywhere in this question.


In [ ]:
...

external_results.head(12).style.format({column: '{:.2%}' for column in ['cv_accuracy', 'closed_accuracy', 'open_accuracy', 'mean_external_accuracy']})

In [ ]:
grader.check("q5_2")

In [ ]:
accuracy_plot = (
    external_results
    .set_index(['recipe', 'model'])[['closed_accuracy', 'open_accuracy']]
    .sort_values('closed_accuracy')
)
accuracy_plot.tail(15).plot.barh(figsize=(11, 9))
plt.xlabel('Accuracy')
plt.xlim(0.75, 1.01)
plt.title('External accuracy for the strongest model/feature combinations')
plt.legend(['Closed-door period', 'Open-door period'])
plt.tight_layout()


### Inspect the mistakes of the CV winner

The next cell chooses the one single classifier with the highest validation accuracy and displays its confusion
matrices.


In [ ]:
cv_winner_row = frozen_choices.loc[frozen_choices['cv_accuracy'].idxmax()]
cv_winner = frozen_searches[cv_winner_row['model']]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test_closed,
    cv_winner.predict(X_test_closed),
    display_labels=['unoccupied', 'occupied'],
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title('Closed-door period')

ConfusionMatrixDisplay.from_predictions(
    y_test_open,
    cv_winner.predict(X_test_open),
    display_labels=['unoccupied', 'occupied'],
    colorbar=False,
    ax=axes[1],
)
axes[1].set_title('Open-door period')
fig.suptitle(f"CV winner: {cv_winner_row['model']} / {cv_winner_row['recipe']}")
fig.tight_layout()


**What do you notice about the test set confusion matrices?**

## 6. What did we reproduce? 📝

Use `cv_results`, `frozen_test_results`, `external_results`, the feature
importance plot, and the confusion matrices to answer these questions with
specific numerical evidence:

1. Did timestamp information help all three classifiers?
2. What happened when light was removed?
3. Could one or two sensors match the accuracy from all five sensors?
4. Did all three algorithms prefer the same feature recipe?
5. Did the model with the highest shuffled CV score transfer best?
6. Which errors would matter most if the model controlled lighting or HVAC?

Then write a 150–250 word conclusion containing:

- The strongest result.
- One conclusion from the paper that your results support.
- One result that depended on the classifier or test period.
- A clear distinction between direct replication and extension.
- One limitation and one proposed follow-up study.

Remember: the minute-level rows are strongly related to neighboring rows.
Shuffled cross-validation can therefore look more optimistic than evaluation
on a separate week. Success in this one office also does not prove that the
same model will work in another room or building.


## Finish Line 🏁

In this lab, you:

- Translated a published study into a scikit-learn experiment.
- Preserved designated external test periods.
- Put timestamp engineering and scaling inside pipelines.
- Tuned logistic regression, kNN, and random forest with `GridSearchCV`.
- Compared sensor subsets without using test results to make model choices.
- Saw why excellent shuffled-CV accuracy does not guarantee equally strong
  transfer to a different measurement period.

Before finishing, make sure every coding question passes.


In [ ]:
# For your convenience, you can run this cell to run all the tests at once!
grader.check_all()
